# 04 — Orchestrated Workflow

**Route, fan out, chain, refine — with observability.**

The four orchestration primitives each do one thing:

- `Parallel` — run steps concurrently and merge.
- `RefinementLoop` — iterate a step until an evaluator passes.
- `Pipeline` — chain steps, each output feeding the next.
- `Router` — pick one path based on the input.

They compose: a `Router` can route to a `Pipeline`, a `Pipeline` step can be a `Parallel`. This notebook builds a small research-brief workflow that nests all four, then observes the run with a `PipelineEvent` stream and a `Workbench`'s `JSONTracer`.

**Prerequisites:** a `.env` with `LLM_API_KEY` and `LLM_MODEL`.

## 1. Load config

In [1]:
from orqest import load_config

config = load_config()
print(f"Model: {config.llm_model}")

Model: openai:gpt-5.2


## 2. The cast

A handful of tiny agents that share one output type. Each is told to be brief — orchestration is about *flow*, not prose.

In [2]:
from pydantic import BaseModel
from orqest.agents import BaseAgent, GlobalState


class Text(BaseModel):
    text: str


def make_agent(name: str, role: str) -> BaseAgent:
    """Build a tiny single-output agent with a brief-by-instruction prompt."""
    class _Agent(BaseAgent[GlobalState, Text]):
        async def _run_implementation(self, state: GlobalState, **kwargs) -> Text:
            result = await self.call_model(state.get_latest_message("user"), state)
            return result.output

    return _Agent(
        agent_name=name,
        system_prompt=f"{role} Respond in 2-3 sentences, no preamble.",
        output_type=Text,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )


tech_agent = make_agent("tech_angle", "You explain the *technical* angle of a topic.")
practical_agent = make_agent("practical_angle", "You explain the *practical, day-to-day* angle of a topic.")
synthesizer = make_agent("synthesizer", "You merge the angles you are given into one tight brief.")
refiner = make_agent("refiner", "You tighten prose, preserving the key point.")
quick_agent = make_agent("quick", "You give a one-paragraph quick take on a topic.")
print("5 agents ready.")

5 agents ready.


## 3. `Parallel` — fan out and merge

`Parallel` launches each step as its own task, waits for all of them, and hands the successful outputs to a merge strategy. Here two agents take different angles on the same topic, concurrently.

In [3]:
from orqest.orchestration import Parallel, MergeStrategy

TOPIC = "vector databases"

fanout = Parallel([tech_agent, practical_agent], merge=MergeStrategy.collect_all)
par_result = await fanout.run(TOPIC)

for angle in par_result.outputs:
    print(f"- {angle.text}\n")

- Vector databases store and index high-dimensional embedding vectors (plus metadata) to support fast approximate nearest-neighbor search using structures like HNSW graphs, IVF/IVF-PQ, or LSH, typically under cosine/dot/L2 metrics. They’re used for semantic retrieval (RAG), similarity search, and recommendations by embedding inputs with an ML model, querying k-NN, and optionally applying metadata filters and re-ranking.

- Vector databases store data as embeddings (number vectors) so you can quickly retrieve “most similar” items—useful day-to-day for semantic search, recommendations, and RAG chatbots that need to pull relevant docs by meaning, not keywords. In practice you generate embeddings for your text/images, upsert them with metadata, then query with a new embedding to get top‑K matches and filter by metadata (date, user, product) before sending results to your app/LLM.



## 4. `RefinementLoop` — iterate until it's tight enough

The loop runs a step, evaluates the output, and uses `state_updater` to build the next input — until the evaluator passes, or `max_iterations` is hit. Here the evaluator is a simple word-count gate; the `refiner` agent tightens the draft each pass.

In [4]:
from orqest.orchestration import RefinementLoop
from orqest.orchestration.loop import EvalResult

WORDY = (
    "Vector databases are, in essence, a specialised kind of database that has been designed and optimised specifically for the storage and the retrieval of high-dimensional vector embeddings, which are the numerical representations that machine learning models produce, and they support similarity search which is the operation of finding the stored vectors that are closest to a given query vector."
)


def evaluator(output: Text) -> EvalResult:
    words = len(output.text.split())
    return EvalResult(passed=words <= 45, feedback=f"{words} words", score=float(words))


def state_updater(current_input: str, output: Text, ev: EvalResult) -> str:
    return f"Tighten this to under 45 words ({ev.feedback} now):\n\n{output.text}"


loop = RefinementLoop(refiner, evaluator, state_updater=state_updater, max_iterations=3)
loop_result = await loop.run(WORDY)

print(f"exit_reason: {loop_result.exit_reason}  ({loop_result.iterations} iterations)")
for rec in loop_result.history:
    print(f"  iter {rec.iteration}: {rec.eval_result.feedback}  passed={rec.eval_result.passed}")
print(f"\nfinal: {loop_result.output.text}")

exit_reason: passed  (1 iterations)
  iter 1: 29 words  passed=True

final: Vector databases are specialised systems for storing and retrieving high‑dimensional vector embeddings produced by ML models. They’re optimised for similarity search—finding stored vectors closest to a given query vector.


## 5. `Pipeline` — chain the stages

A `Pipeline` runs steps in sequence, feeding each output into the next. Steps are auto-coerced: a `BaseAgent` becomes an `AgentStep`, a plain async callable becomes a `FunctionStep`. That lets us drop the `Parallel` fan-out in as stage one — wrapped in a thin callable that adapts its `ParallelResult` into the string the synthesizer wants.

`run_stream` yields a `PipelineEvent` at every lifecycle point — the pipeline's built-in observability.

In [5]:
from orqest.orchestration import Pipeline


async def fan_out_stage(topic: str) -> str:
    """Stage 1: run the Parallel fan-out, format its outputs for the next stage."""
    result = await Parallel([tech_agent, practical_agent]).run(topic)
    tech, practical = result.outputs
    return (
        f"TECHNICAL ANGLE: {tech.text}\n\n"
        f"PRACTICAL ANGLE: {practical.text}\n\n"
        "Merge these into one tight brief."
    )


pipeline = Pipeline([fan_out_stage, synthesizer], name="research_brief")

events = []
async for ev in pipeline.run_stream(TOPIC):
    events.append(ev)
    print(f"{ev.event_type:18s} step={ev.step_name}")

brief = events[-1].data["output"]
print(f"\nbrief: {brief.text}")

pipeline_start     step=
step_start         step=fan_out_stage


step_complete      step=fan_out_stage
step_start         step=synthesizer


step_complete      step=synthesizer
pipeline_complete  step=

brief: A vector database stores high‑dimensional embedding vectors (with metadata and CRUD/upsert) and enables fast approximate nearest‑neighbor similarity search via indexes like HNSW or IVF/PQ using cosine/dot/L2 metrics. Practically, you embed items, upsert the vectors plus metadata, then query with a new embedding—optionally combining keyword/metadata filters for hybrid retrieval—to quickly return semantically similar results at million‑scale for semantic search, recommendations, and RAG chatbots.


## 6. `Router` — pick the path, inside a `Workbench`

The `Router` evaluates each route's condition in order; the first match wins, and anything unmatched falls through to the `fallback`. Here a "deep" request runs the full `Pipeline`; everything else gets the `quick_agent`.

We run it inside a `Workbench` and bracket each call in a `JSONTracer` span — the `Workbench`'s tracer is the observability seam for work that isn't a compound-tool flow.

In [6]:
import tempfile, os
from orqest.orchestration import Router, Route
from orqest.workbench import Workbench
from orqest.memory import LocalMemoryStore


async def deep_route(request: str):
    return await pipeline.run(request)


router = Router(
    [Route("deep", deep_route, condition=lambda r: "deep" in r.lower())],
    fallback=quick_agent,
    name="brief_router",
)

wb = Workbench(memory=LocalMemoryStore(os.path.join(tempfile.mkdtemp(), "wf.db")))

span = wb.tracer.start_span("deep_request", agent_name="brief_router")
deep_out = await router.run("Please do a deep brief on vector databases.")
wb.tracer.end_span(span, status="ok")

span = wb.tracer.start_span("quick_request", agent_name="brief_router")
quick_out = await router.run("Quick take on vector databases?")
wb.tracer.end_span(span, status="ok")

print(f"deep  -> {deep_out.text[:110]}")
print(f"quick -> {quick_out.text[:110]}")

deep  -> A vector database stores high‑dimensional embedding vectors and uses approximate nearest‑neighbor (ANN) indexe
quick -> Vector databases store and index high‑dimensional embeddings so you can do fast similarity search (nearest nei


In [7]:
# The Workbench's tracer recorded a span per request.
for s in wb.tracer.export_json():
    print(f"{s['name']:16s} {s['status']:4s} {s['duration_ms']:.0f} ms")

deep_request     ok   7531 ms
quick_request    ok   3487 ms


## What's happening under the hood

- **Auto-coercion** is what makes the primitives compose. `Pipeline` / `Parallel` / `Router` accept `StepLike = Step | BaseAgent | callable`; `_coerce_step` wraps a `BaseAgent` in an `AgentStep` (fresh `GlobalState` per call) and a callable in a `FunctionStep`. That's why a bare `async def` and a `BaseAgent` sit side by side in the same step list.
- The adapter callables (`fan_out_stage`, `deep_route`) exist because each primitive returns its *own* result type — `ParallelResult`, `LoopResult`, a raw `OutputT`. A two-line callable bridges between them. Nesting is composition, not inheritance.
- `Pipeline.run_stream` is the pipeline's native observability — a typed `PipelineEvent` for `pipeline_start` / `step_start` / `step_complete` / `step_skip` / `step_error` / `pipeline_complete`. `Pipeline.run` is just `run_stream` drained to the final output.
- Per-step error handling lives in `StepConfig` — pass `(step, StepConfig(on_error=ErrorStrategy.SKIP))` to skip a failing step instead of aborting the pipeline.
- The `Workbench`'s `JSONTracer` is the seam for tracing work that isn't a compound-tool flow (orchestration primitives don't fire the `HookRunner` lifecycle — that's for `CompoundTool` / `SubAgentTool`). `start_span` / `end_span` bracket whatever you want to time.